# AMRS (Adaptive Model Recovery System) — Colab GPU Bootstrap (INF-2.2)

This notebook bootstraps the full AMRS stack on a Google Colab GPU runtime (T4, V100, or A100):
1. **Optional Google Drive mount** for persistent artifact/checkpoint storage across sessions.
2. **Repository checkout & editable package installation** with CUDA verification.
3. **Synthetic scenario generation** (`s1_overfit` through `s6_healthy`) running on GPU.
4. **Artifact caching to Google Drive**.
5. **FastAPI backend launch** on private internal port `8000` (`http://127.0.0.1:8000`).
6. **Next.js frontend launch** on port `3000` with internal `/api/*` proxy rewrite to FastAPI.
7. **Cloudflared / Localtunnel ingress** exposing **only** port 3000 to public browser traffic.

> **Security Architecture Notice**:
> FastAPI is **never** exposed directly to the public internet. Port 8000 is bound strictly to `127.0.0.1`.
> Browser requests pass exclusively through the Next.js same-origin `/api` proxy.

### Step 1: Optional Google Drive Mount
Mount Google Drive to persist generated scenarios, checkpoints, and benchmark results across session restarts.

In [ ]:
import os
from pathlib import Path

MOUNT_DRIVE = True  # Set to False if you want purely ephemeral scratch storage
DRIVE_PERSISTENCE_DIR = Path("/content/drive/MyDrive/amrs_cache")

if MOUNT_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        DRIVE_PERSISTENCE_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Google Drive mounted. Cache directory ready at: {DRIVE_PERSISTENCE_DIR}")
    except Exception as e:
        print(f"Drive mount skipped or unavailable: {e}")
        MOUNT_DRIVE = False

### Step 2: Verify GPU Accelerator & PyTorch CUDA

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    print(f"GPU Device: {device_name}")
    !nvidia-smi
else:
    print("WARNING: No GPU detected. AMRS will run in CPU fallback mode.")
    print("To enable GPU, go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")

### Step 3: Clone Repository and Install AMRS Package

In [ ]:
# Clone if not already present in Colab workspace
if not os.path.exists("AMRS"):
    !git clone https://github.com/manthanrailkar/AMRS.git
    %cd AMRS
else:
    %cd AMRS

!pwd
# Install AMRS and dependencies in editable mode
!pip install --upgrade pip
!pip install -e .
!amrs --version

### Step 4: Build Scenarios on GPU & Cache to Google Drive

In [ ]:
import shutil

device_flag = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Building benchmark scenarios with device={device_flag}...")

# Check if cached artifacts exist in Google Drive to skip rebuilding
cached_scenarios = DRIVE_PERSISTENCE_DIR / "scenarios" if MOUNT_DRIVE else None
if cached_scenarios and cached_scenarios.exists() and any(cached_scenarios.iterdir()):
    print("Restoring cached scenarios from Google Drive...")
    shutil.copytree(cached_scenarios, "artifacts/scenarios", dirs_exist_ok=True)
else:
    !amrs scenarios build --all --device {device_flag}
    if MOUNT_DRIVE and os.path.exists("artifacts/scenarios"):
        print("Caching built scenarios to Google Drive...")
        shutil.copytree("artifacts/scenarios", cached_scenarios, dirs_exist_ok=True)

print("Scenarios ready in artifacts/scenarios:")
!ls -lah artifacts/scenarios

### Step 5: Start Private FastAPI Service (Port 8000)
FastAPI is bound strictly to `127.0.0.1:8000` and serves `/api/v1/health` and recovery endpoints.

In [ ]:
import subprocess
import time

print("Starting FastAPI backend on 127.0.0.1:8000...")
backend_proc = subprocess.Popen(
    ["uvicorn", "amrs.api.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

# Wait for server startup
time.sleep(3)

# Healthcheck probe on internal service
!curl -s -f http://127.0.0.1:8000/api/v1/health || echo "FastAPI initialization pending..."
print("\nFastAPI backend is running privately on port 8000.")

### Step 6: Start Next.js Frontend (Port 3000)
Next.js proxies `/api/*` requests internally to `AMRS_API_INTERNAL_URL=http://127.0.0.1:8000`.

In [ ]:
import os
import subprocess
import time

# Navigate to Next.js app directory
%cd apps/web

# Install frontend dependencies
!npm ci || npm install

# Build Next.js production bundle
!npm run build

# Start Next.js server on port 3000 with internal backend URL wired
frontend_env = os.environ.copy()
frontend_env["AMRS_API_INTERNAL_URL"] = "http://127.0.0.1:8000"
frontend_env["PORT"] = "3000"

web_proc = subprocess.Popen(
    ["npm", "run", "start", "--", "-p", "3000"],
    env=frontend_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

time.sleep(5)
%cd ../..
print("Next.js frontend listening on 127.0.0.1:3000.")

### Step 7: Expose ONLY Next.js (Port 3000) via Public Tunnel
Cloudflared connects to `http://127.0.0.1:3000`. The public tunnel URL provides direct access to the Next.js UI.

In [ ]:
# Install Cloudflared CLI if not already installed
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1 || true

print("=" * 60)
print("ESTABLISHING SECURE CLOUDFLARE TUNNEL (PORT 3000 ONLY)")
print("=" * 60)
print("Access the AMRS Web Dashboard at the trycloudflare.com link below:")

# Launch Cloudflared tunnel attached strictly to port 3000
!cloudflared tunnel --url http://127.0.0.1:3000